# Paper B batch 5 — CoNLL-2003: Li-style MRC vs a *tuned* BIO tagger (6.22, preregistered)
Spec, predictions and decision rules: `experiments/rigor/PREREG_6.22_conll_tuned_bio.md` (committed before any run — do not edit after results). Encoders: bert-base-cased (WordPiece), roberta-base (byte-BPE), xlm-roberta-base (SentencePiece). Arms: A BIO ow=1.0, B BIO best-ow (dev-selected), C = B with BIO-valid Viterbi decode, D MRC best pos-weight. LR fixed 3e-5, 3 epochs, seeds 42/123/7; sweep on seed 42 only. CoNLL is pre-tokenized, so the SentencePiece punctuation artifact cannot occur here.
Runtime → **A100**. Cell 4 is a timing dry-run: read it before launching cells 5–7. Every run skips if its `metrics.json` already exists on Drive (disconnect-safe: just rerun). No checkpoints are saved. Last cell releases the runtime.

In [ ]:
# 1. Setup: Drive + repo + deps + GPU gate
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt
import torch
assert torch.cuda.is_available(), 'CPU-only runtime (happens after restarts) — Runtime > Change runtime type > GPU'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: scripts pushed, Drive root durable + writable. T0 proves runs are fresh.
import os, sys, json, glob, time, shutil
T0 = time.time()
for f in ('run_39_conll_qa_vs_bio.py','run_40_conll_aggregate.py','PREREG_6.22_conll_tuned_bio.md'):
    assert os.path.exists(f'experiments/rigor/{f}'), f'{f} missing — push it first'
sys.path.insert(0, 'experiments/rigor')
from run_40_conll_aggregate import OW, PW, select
ROOT = '/content/drive/MyDrive/IdiomatorRigor_NER_conll'
os.makedirs(ROOT, exist_ok=True)
assert os.path.realpath(ROOT).startswith('/content/drive'), 'ROOT is not on Drive'
open(f'{ROOT}/.probe','w').write('ok'); assert open(f'{ROOT}/.probe').read()=='ok'; os.remove(f'{ROOT}/.probe')
ENCS = [('bert-base-cased','bert'), ('roberta-base','roberta'), ('xlm-roberta-base','xlmr')]
print('preflight ok; OW', OW, 'PW', PW)

In [ ]:
# 3. CoNLL-2003 data (HF datasets-server, ~8 min) — cached on Drive so a restart does not refetch.
if os.path.exists(f'{ROOT}/_data/train.jsonl') and not os.path.exists('data/conll03/train.jsonl'):
    shutil.copytree(f'{ROOT}/_data', 'data/conll03')
!python experiments/rigor/run_39_conll_qa_vs_bio.py --fetch-only
n = {s: sum(1 for _ in open(f'data/conll03/{s}.jsonl')) for s in ('train','dev','test')}
assert n == {'train': 14041, 'dev': 3250, 'test': 3453}, n
shutil.copytree('data/conll03', f'{ROOT}/_data', dirs_exist_ok=True)
print('data ok', n)

In [ ]:
# 4. Timing dry-run: 800 sentences, 1 epoch, bert. Extrapolate to the full run BEFORE launching 5-7.
for mode in ('bio','mrc'):
    !python experiments/rigor/run_39_conll_qa_vs_bio.py --mode {mode} --model bert-base-cased --max-n 800 --epochs 1 --out-dir /content/_dry_{mode} --force
    s = json.load(open(f'/content/_dry_{mode}/metrics.json'))['seconds']
    print(f'### {mode}: {s}s for 800 train sentences x 1 epoch (+ eval on 800 dev/test) -> ~{s*(14041/800)*3/60:.0f} min per full 3-epoch run (upper bound; eval share shrinks)')

## Sweep (seed 42): BIO O-weight {1.0, 0.5, 0.2, 0.1} and MRC positive-weight {1, 5}, per encoder — 18 runs

In [ ]:
# 5. Sweep. Resumable.
for model, short in ENCS:
    for w in OW:
        !python experiments/rigor/run_39_conll_qa_vs_bio.py --mode bio --model {model} --seed 42 --o-weight {w} --out-dir {ROOT}/{short}_bio_ow{w}_s42
    for p in PW:
        !python experiments/rigor/run_39_conll_qa_vs_bio.py --mode mrc --model {model} --seed 42 --pos-weight {p} --out-dir {ROOT}/{short}_mrc_pw{p}_s42

In [ ]:
# 6. Select per encoder on seed-42 DEV F1 (never test). Saved to Drive so a restart reuses it.
SEL = {short: select(ROOT, short) for _, short in ENCS}
for short, (ow, pw) in SEL.items():
    print(short, 'best O-weight', ow, '| best MRC pos-weight', pw)
json.dump(SEL, open(f'{ROOT}/selection.json','w'))

In [ ]:
# 7. Finals: seeds 123 and 7. BIO at ow=1.0 (arm A) and at the selected ow (arms B/C); MRC at the selected pos-weight (arm D). Resumable.
for model, short in ENCS:
    ow, pw = SEL[short]
    for s in (123, 7):
        for w in sorted({'1.0', ow}):
            !python experiments/rigor/run_39_conll_qa_vs_bio.py --mode bio --model {model} --seed {s} --o-weight {w} --out-dir {ROOT}/{short}_bio_ow{w}_s{s}
        !python experiments/rigor/run_39_conll_qa_vs_bio.py --mode mrc --model {model} --seed {s} --pos-weight {pw} --out-dir {ROOT}/{short}_mrc_pw{pw}_s{s}

In [ ]:
# 8. Aggregate: arms A/B/C/D, paired bootstrap, TOST, PREREG verdict. Written to Drive.
import subprocess
r = subprocess.run(['python','experiments/rigor/run_40_conll_aggregate.py','--root',ROOT,'--encs',*[s for _, s in ENCS],'--out',f'{ROOT}/run_40_aggregate.json'],capture_output=True,text=True)
print(r.stdout, r.stderr[-1500:])
assert r.returncode == 0

In [ ]:
# FINAL: prove everything is on Drive, flush, then release the runtime.
# "Run all" stops at the first error, so this only runs if every cell above succeeded.
need = []
for _, short in ENCS:
    ow, pw = SEL[short]
    need += [f'{short}_bio_ow{w}_s42' for w in OW] + [f'{short}_mrc_pw{p}_s42' for p in PW]
    for s in (123, 7):
        need += [f'{short}_bio_ow{w}_s{s}' for w in sorted({'1.0', ow})] + [f'{short}_mrc_pw{pw}_s{s}']
missing = [d for d in need if not (os.path.exists(f'{ROOT}/{d}/metrics.json') and os.path.getsize(f'{ROOT}/{d}/test_predictions.jsonl') > 0)]
assert not missing, f'{len(missing)} runs not persisted to Drive, NOT disconnecting: {missing[:5]}'
assert os.path.getmtime(f'{ROOT}/run_40_aggregate.json') > T0, 'aggregate is stale'
print('✓ all expected runs persisted on Drive')
drive.flush_and_unmount()   # sync pending Drive writes before the VM disappears
print('Drive flushed — releasing runtime')
from google.colab import runtime
runtime.unassign()